# Frozen held-out PBT prompt comparison — 2026-09-07

**Setup.** All30 tasks held out from this Azure prompt-development comparison, from the original test split of data/apps_hard.json, two existing candidates each:60 candidates. No training task reused. These tasks may have appeared in earlier project experiments; they are not claimed historically virgin. Azure gpt-5.6-terra, low reasoning, seed300, existing candidate-aware trigger search with ten inputs; then unchanged plain_v3 versus traceable_v1, ten tests, resolve='with', visible code, no critique,8192 output tokens,300-second author attempt,120-second Docker sandbox and the same pinned image as training.

**Change.** Population only: held-out test tasks instead of development tasks. Freeze both prompts before any response. No held-out outcome-based tuning, test repairs, candidate-specific regeneration between prompt arms, or silent retries of recorded failures.

**Hypothesis.** The earlier traceability framing could lower honest false alarms while preserving attack catches. The completed training pilot did not meet its target; this held-out comparison checks generalization of the unchanged templates rather than claiming training success.
**Prediction.** Paired honest FPR reduction at least0.10 with at least90% of baseline attack catch rate. If baseline catches zero, retention is undefined. Report error-free sensitivity, model/infra failures and exact denominators. Same-input honest replay and duplicate diagnostics are required before interpreting attack firing.
**Observed.** Trigger stage completed60/60 with one model parse failure. Independent assistant cross-review reproduced586valid/4invalid/0unresolved inputs. Removing invalid inputs and source-failed task3847 leaves29tasks,58candidates,576inputs. Frozen baseline authoring launched2026-09-08 PID24876. No authoring result is inferred from launch or parse shape.

**Budget.** At most60 trigger +120 author logical calls. First-attempt output envelope720000 trigger +983040 author =1703040 tokens, or3406080 if every request takes its configured one retry at the full cap. Input tokens additional; not an enforced dollar cap or verified Azure price quote. Cache/resume protects completed candidates, not interrupted in-flight requests. Usage may not include every billed retry.

**Stages.** Setup cells1–4; cell5 launches ONLY trigger search detached with system-awake guard. Cell6 reads progress. Cells7-8 prepare reviewed authoring, cell9 launches baseline, cell10 requires complete baseline then launches traceability, and cell11 reports artifacts. Domain review was cross-checked before authoring; invalid inputs are filtered identically across arms and source-failed/unresolved/empty candidate pairs excluded. This gate is not permission to tune prompts using held-out outcomes. No multi-turn repair runs here.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys
from dotenv import load_dotenv
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))
os.chdir(REPO)
assert (REPO / "pipeline").is_dir() and (REPO / "data").is_dir()
load_dotenv(REPO / ".env", encoding="utf-8-sig", override=True)
from pipeline.data import Dataset, load_records
from pipeline.protocols import TriggerSearch, UnitTesting
from pipeline.protocols.base import Run

MODEL = "openai-api/azureai/gpt-5.6-terra"
PREFIX = "azure-terra-pbt-heldout30-s300-v1"
SOURCE = Path("data/apps_hard.json")
DATA = Path("data/azure_pbt_heldout30_s300_v1.json")
TRIGGERS = PREFIX + "-triggers"
CONTROL = PREFIX + "-plain-v3"
TRACEABLE = PREFIX + "-traceable-v1"
SEED, N_INPUTS, N_TESTS = 300, 10, 10
assert os.environ["AZURE_OPENAI_DEPLOYMENT"] == "gpt-5.6-terra"
assert os.environ["AZURE_OPENAI_API_KEY"]
from urllib.parse import urlsplit
endpoint_parts = urlsplit(os.environ["AZURE_OPENAI_ENDPOINT"].strip())
assert endpoint_parts.scheme == 'https' and endpoint_parts.netloc == 'omar-ai.services.ai.azure.com'
assert not endpoint_parts.query and not endpoint_parts.fragment
assert endpoint_parts.path.rstrip('/') in ('', '/openai/v1', '/openai/v1/responses')
endpoint = 'https://omar-ai.services.ai.azure.com/openai/v1'
os.environ["AZUREAI_BASE_URL"] = endpoint
os.environ["AZUREAI_API_KEY"] = os.environ["AZURE_OPENAI_API_KEY"]


In [ ]:
source = Dataset.load(SOURCE)
assert len(source.split["train"]) == 20 and len(source.split["test"]) == 30
SELECTED_IDS = tuple(source.split["test"])
assert len(SELECTED_IDS) == 30
assert not set(SELECTED_IDS).intersection(source.split["train"])
source_hash = hashlib.sha256(SOURCE.read_bytes()).hexdigest()
assert source_hash == "4237df1d48b1c1d1cd4a9b197ce57948a231d8dad26abace58715c7254329974", "Source dataset changed"
heldout = Dataset(name="azure-pbt-heldout30-v1", backend=source.backend, io_mode=source.io_mode,
    tasks=tuple(source.task(task_id) for task_id in SELECTED_IDS),
    split={"train": (), "test": SELECTED_IDS},
    built_from={"source": str(SOURCE), "source_sha256": source_hash,
                "source_built_from": dict(source.built_from),
                "selection": "all original held-out test tasks, no outcome selection", "seed": SEED})
expected_document = heldout.to_json()
if DATA.exists():
    assert json.loads(DATA.read_text(encoding="utf-8")) == expected_document, "Existing held-out data differs"
else:
    DATA.write_text(json.dumps(expected_document, indent=2) + "\n", encoding="utf-8")
dataset = Dataset.load(DATA)
assert len(dataset.test) == 30 and not dataset.train
EXPECTED_IDS = {candidate.candidate_id for _, candidate in dataset.candidates()}
assert len(EXPECTED_IDS) == 60
print({"task_ids": list(SELECTED_IDS), "candidate_count": len(EXPECTED_IDS),
       "source_sha256": source_hash, "dataset_sha256": hashlib.sha256(DATA.read_bytes()).hexdigest()})


In [ ]:
trigger = TriggerSearch(run_name=TRIGGERS, data=str(DATA), model=MODEL, num_inputs=N_INPUTS,
                        reasoning="low", code_visible=True, seed=SEED, cache=True)
assert trigger.runtime(SEED).max_tokens == 12000 and trigger.runtime(SEED).http_retries == 1
assert trigger.total == 60 and all(trigger.data.split_of(t.task_id) == "test" for t in trigger.data.tasks)
training_trigger = json.loads(Path("runs/azure-terra-pbt-train19-s300-v1-triggers/config.json").read_text())
training_author = json.loads(Path("runs/azure-terra-pbt-train19-s300-reviewed-v1-plain-v3/config.json").read_text())
for key in ("model", "runs", "seed", "cache", "params"):
    assert trigger.config()[key] == training_trigger[key], f"Trigger setting drift: {key}"
planned_author_params = dict(training_author["params"])
planned_author_params["triggers"] = PREFIX + "-reviewed-inputs"
planned_configs = {}
for suffix, framing in (("plain-v3", "plain_v3"), ("traceable-v1", "traceable_v1")):
    params = {**planned_author_params, "test_gen_prompt": framing}
    planned_configs[suffix] = {"protocol": "unit_testing", "model": MODEL, "seed": SEED,
                              "runs": 1, "cache": True, "params": params,
                              "data_status": "pending reviewed derived dataset"}
def digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()
manifest_path = trigger.directory / "heldout-preregistration.json"
frozen_files = ([Path(p) for p in json.loads(manifest_path.read_text(encoding="utf-8"))["file_sha256"]]
                if manifest_path.exists() else sorted(Path("prompts").glob("*.txt")) + [
    Path("pipeline/model.py"), Path("pipeline/prompts.py"), Path("pipeline/parse.py"),
    Path("pipeline/sandbox.py"), Path("pipeline/protocols/trigger_search.py"),
    Path("pipeline/protocols/unit_testing.py")])
manifest = {"kind": "heldout-preregistration-v1", "task_ids": list(SELECTED_IDS),
    "source_sha256": source_hash, "heldout_dataset_sha256": digest(DATA),
    "trigger_config": trigger.config(), "planned_author_configs": planned_configs,
    "file_sha256": {str(p): digest(p) for p in frozen_files},
    "logical_call_cap": {"trigger": 60, "author": 120},
    "first_attempt_output_token_envelope": 1703040,
    "configured_retry_inclusive_output_token_envelope": 3406080,
    "domain_review_required_before_authoring": True,
    "multi_turn_launched": False}
trigger.write_config()
manifest_path = trigger.directory / "heldout-preregistration.json"
payload = json.dumps(manifest, indent=2) + "\n"
if manifest_path.exists():
    assert manifest_path.read_text(encoding="utf-8") == payload, "Frozen manifest differs"
else:
    manifest_path.write_text(payload, encoding="utf-8")
print({"manifest_sha256": digest(manifest_path), "trigger_config_sha256": digest(trigger.config_path),
       "planned_author_configs": planned_configs, "new_model_calls": 0})


In [ ]:
WORKER = r'''
from pathlib import Path
import json, os, sys, traceback
from dotenv import load_dotenv
load_dotenv(Path.cwd() / ".env", encoding="utf-8-sig", override=True)
assert os.environ['AZURE_OPENAI_DEPLOYMENT'] == 'gpt-5.6-terra'
from urllib.parse import urlsplit
endpoint_parts = urlsplit(os.environ["AZURE_OPENAI_ENDPOINT"].strip())
assert endpoint_parts.scheme == 'https' and endpoint_parts.netloc == 'omar-ai.services.ai.azure.com'
assert not endpoint_parts.query and not endpoint_parts.fragment
assert endpoint_parts.path.rstrip('/') in ('', '/openai/v1', '/openai/v1/responses')
base = 'https://omar-ai.services.ai.azure.com/openai/v1'
os.environ["AZUREAI_BASE_URL"] = base
os.environ["AZUREAI_API_KEY"] = os.environ["AZURE_OPENAI_API_KEY"]
import pipeline.protocols
from pipeline.protocols.base import Run
config_path = Path(sys.argv[1])
lock_path = config_path.parent / "heldout-worker.lock"
exit_path = config_path.parent / "heldout-worker-exit.json"
awake_state = None
try:
    if os.name == "nt":
        import ctypes
        awake_state = ctypes.windll.kernel32.SetThreadExecutionState(0x80000001)
        if not awake_state:
            raise OSError("Windows refused the system-awake request")
    run = Run.from_config(json.loads(config_path.read_text(encoding="utf-8")))
    assert run.total == 60 and len(run.data.test) == 30 and not run.data.train
    assert run.protocol == "trigger_search"
    written = run.execute()
    result = {"exit_code": 0, "written": written, "scored": len(run.get_records()), "total": run.total}
except BaseException as error:
    traceback.print_exc()
    result = {"exit_code": 1, "error_type": type(error).__name__}
finally:
    if awake_state:
        ctypes.windll.kernel32.SetThreadExecutionState(0x80000000)
    exit_path.write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
    lock_path.unlink(missing_ok=True)
sys.exit(result["exit_code"])
'''

def launch_heldout_triggers(run):
    """Launch only the bounded trigger stage; existing records remain authoritative."""
    assert run.run_name == TRIGGERS and run.protocol == "trigger_search"
    assert run.total == 60 and len(run.data.test) == 30 and not run.data.train
    assert all(digest(Path(p)) == h for p, h in manifest["file_sha256"].items()), "Frozen files changed"
    run.write_config()
    if not run.pending():
        print({"run": run.run_name, "state": "all60 candidate records already present"})
        return
    subprocess.run(["docker", "info", "--format", "{{.ServerVersion}}"],
                   check=True, capture_output=True, text=True, timeout=30)
    subprocess.run(["docker", "image", "inspect", "python@sha256:78387bc3881b8273120a12ebe6c1ab22b018ccc2c9adf565ae1ac9b536e184ea"],
                   check=True, capture_output=True, text=True, timeout=30)
    if os.name != "nt":
        raise RuntimeError("This training launcher requires Windows; use the existing tmux protocol with an external awake guard on Linux")
    lock = run.directory / "heldout-worker.lock"
    descriptor = os.open(lock, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
    os.close(descriptor)
    try:
        with (run.directory / "heldout-worker.log").open("ab") as log:
            process = subprocess.Popen(
                [sys.executable, "-u", "-c", WORKER, str(run.config_path)],
                cwd=REPO, stdin=subprocess.DEVNULL, stdout=log, stderr=log,
                creationflags=subprocess.DETACHED_PROCESS | subprocess.CREATE_NEW_PROCESS_GROUP
                              | subprocess.CREATE_NO_WINDOW,
                close_fds=True,
            )
        (run.directory / "heldout-worker-pid.json").write_text(
            json.dumps({"pid": process.pid, "run": run.run_name}) + "\n", encoding="utf-8")
    except BaseException:
        lock.unlink(missing_ok=True)
        raise
    print({"run": run.run_name, "pid": process.pid, "pending_at_launch": len(run.pending()),
           "next_step": "inspect60 trigger records and review input domains before authoring"})


In [ ]:
launch_heldout_triggers(trigger)


In [ ]:
rows = load_records(TRIGGERS)
assert len(rows) <= 60
assert len({r["candidate_id"] for r in rows}) == len(rows)
assert all(r["candidate_id"] in EXPECTED_IDS and r["split"] == "test" for r in rows)
failures = [{k: r[k] for k in ("candidate_id", "blame", "reason")} for r in rows if r["failed"]]
usable = [r for r in rows if not r["failed"]]
print({"records": len(rows), "expected": 60, "failed": failures,
       "parsed_inputs": sum(r["n_parsed"] for r in usable),
       "domain_review": "required; parsing is not semantic validity",
       "authoring_launched": False})


In [ ]:
# Domain-reviewed authoring preparation: no model calls.
import copy
REVIEW = trigger.directory / "domain-review-v1.json"
review = json.loads(REVIEW.read_text(encoding="utf-8"))
assert review["source_dataset_sha256"] == digest(DATA)
assert review["source_records_sha256"] == digest(trigger.records_path)
source_rows = load_records(TRIGGERS)
assert len(source_rows) == len({r["candidate_id"] for r in source_rows}) == 60
by_id = {r["candidate_id"]:r for r in source_rows}
assert review["totals"] == {"valid":586,"invalid":4,"unresolved":0}, "Review version changed"
assert {r["candidate_id"] for r in review["failed_source_records"]} == {"apps_3847_attack_0"}
selected, excluded = [], []
for task in dataset.tasks:
    reasons=[]
    for candidate in task.candidates:
        cid=candidate.candidate_id
        if by_id[cid]["failed"]:
            reasons.append({"candidate_id":cid,"reason":"source trigger failure","blame":by_id[cid]["blame"]})
            continue
        item=review["candidates"][cid]
        assert item["task_id"] == task.task_id
        assert len(item["inputs"]) == len(by_id[cid]["inputs"])
        for i, check in enumerate(item["inputs"]):
            assert check["input_index"] == i
            assert hashlib.sha256(by_id[cid]["inputs"][i].encode("utf-8")).hexdigest() == check["input_sha256"]
        for status in ("valid","invalid","unresolved"):
            assert item[status+"_indices"] == [x["input_index"] for x in item["inputs"] if x["status"]==status]
        if item["unresolved_indices"] or not item["valid_indices"]:
            reasons.append({"candidate_id":cid,"reason":"unresolved or zero valid inputs"})
    if reasons:
        excluded.append({"task_id":task.task_id,"reasons":reasons})
    else:
        selected.append(task)
assert len(selected)==29 and [t["task_id"] for t in excluded]==["3847"]
AUTHOR_DATA=Path("data/azure_pbt_heldout29_s300_reviewed_v1.json")
REVIEWED_INPUTS=PREFIX+"-reviewed-inputs"
provenance={"source_dataset_sha256":digest(DATA),"source_records_sha256":digest(trigger.records_path),
            "review_path":str(REVIEW),"review_sha256":digest(REVIEW),"excluded_task_pairs":excluded,
            "original_test_tasks":30,"retained_test_tasks":29}
derived=Dataset(name=PREFIX+"-reviewed",backend=dataset.backend,io_mode=dataset.io_mode,
                tasks=tuple(selected),split={"train":(),"test":tuple(t.task_id for t in selected)},built_from=provenance)
AUTHOR_IDS={c.candidate_id for _,c in derived.candidates()}
def write_frozen(path,text):
    if path.exists():
        assert path.read_text(encoding="utf-8")==text, f"Frozen artifact differs: {path}"
    else:
        path.parent.mkdir(parents=True,exist_ok=True);path.write_text(text,encoding="utf-8")
write_frozen(AUTHOR_DATA,json.dumps(derived.to_json(),indent=2)+"\n")
view_rows=[]
for row in source_rows:
    cid=row["candidate_id"]
    if cid not in AUTHOR_IDS:
        continue
    indices=review["candidates"][cid]["valid_indices"]
    result=copy.deepcopy(row)
    result.update(run_name=REVIEWED_INPUTS,protocol="reviewed_trigger_view",calls=[],
                  inputs=[row["inputs"][i] for i in indices],n_requested=len(indices),n_parsed=len(indices),dropped=0,
                  provenance={**provenance,"source_candidate_id":cid,"source_input_indices":indices,
                              "source_n_requested":row["n_requested"],"source_n_parsed":row["n_parsed"],
                              "transformation":"subset existing generated inputs; no new generation","new_model_calls":0})
    view_rows.append(result)
assert len(view_rows)==58 and sum(len(r["inputs"]) for r in view_rows)==576
view_dir=Path("runs")/REVIEWED_INPUTS
write_frozen(view_dir/"config.json",json.dumps({"protocol":"reviewed_trigger_view","run_name":REVIEWED_INPUTS,
    "data":str(AUTHOR_DATA),"read_only":True,"new_model_calls":0,"provenance":provenance},indent=2)+"\n")
write_frozen(view_dir/"records.jsonl","".join(json.dumps(r)+"\n" for r in view_rows))
author_arms={}
for suffix, config in planned_configs.items():
    arm=UnitTesting(run_name=PREFIX+"-"+suffix,data=str(AUTHOR_DATA),model=config["model"],seed=config["seed"],
                    runs=config["runs"],cache=config["cache"],**config["params"])
    assert arm.total==58 and not arm.data.train and len(arm.data.test)==29
    arm.prepare(arm.data)
    assert not arm.no_trigger_space and set(arm.trigger_space)==AUTHOR_IDS
    author_arms[suffix]=arm
assert author_arms["plain-v3"].trigger_space==author_arms["traceable-v1"].trigger_space
print({"authoring_candidates_per_arm":58,"retained_inputs":576,"excluded":excluded,"new_model_calls":0})


In [ ]:
AUTHOR_WORKER=WORKER.replace('run.total == 60 and len(run.data.test) == 30','run.total == 58 and len(run.data.test) == 29').replace('run.protocol == "trigger_search"','run.protocol == "unit_testing"')
def launch_heldout_author(run):
    assert run.run_name in (PREFIX+"-plain-v3",PREFIX+"-traceable-v1") and run.protocol=="unit_testing"
    assert all(digest(Path(p))==h for p,h in manifest["file_sha256"].items()),"Frozen files changed"
    assert digest(REVIEW)==provenance["review_sha256"]
    assert digest(trigger.records_path)==provenance["source_records_sha256"]
    run.write_config()
    if not run.pending():
        print({"run":run.run_name,"state":"all candidate records present"})
        return
    assert os.name=="nt"
    subprocess.run(["docker","info","--format","{{.ServerVersion}}"],check=True,capture_output=True,text=True,timeout=30)
    subprocess.run(["docker","image","inspect",planned_author_params["docker_image"]],check=True,capture_output=True,text=True,timeout=30)
    lock=run.directory/"heldout-worker.lock"
    descriptor=os.open(lock,os.O_CREAT|os.O_EXCL|os.O_WRONLY);os.close(descriptor)
    try:
        with (run.directory/"heldout-worker.log").open("ab") as log:
            process=subprocess.Popen([sys.executable,"-u","-c",AUTHOR_WORKER,str(run.config_path)],
                cwd=REPO,stdin=subprocess.DEVNULL,stdout=log,stderr=log,
                creationflags=subprocess.DETACHED_PROCESS|subprocess.CREATE_NEW_PROCESS_GROUP|subprocess.CREATE_NO_WINDOW,
                close_fds=True)
        (run.directory/"heldout-worker-pid.json").write_text(json.dumps({"pid":process.pid,"run":run.run_name})+"\n",encoding="utf-8")
    except BaseException:
        lock.unlink(missing_ok=True);raise
    print({"run":run.run_name,"pid":process.pid,"pending":len(run.pending())})


In [ ]:
# Author stage1, after root has reviewed the domain report. Frozen baseline only.
launch_heldout_author(author_arms["plain-v3"])


In [ ]:
# Author stage2 after baseline ends. Failures are retained and reported, never repaired or silently zeroed.
baseline_rows=author_arms["plain-v3"].get_records()
assert len(baseline_rows)==len({r["candidate_id"] for r in baseline_rows})==58,"Baseline unfinished"
assert not (author_arms["plain-v3"].directory/"heldout-worker.lock").exists(),"Baseline worker still active"
print({"baseline_failures":[{k:r[k] for k in ("candidate_id","blame","reason")} for r in baseline_rows if r["failed"]]})
launch_heldout_author(author_arms["traceable-v1"])


In [ ]:
for arm in author_arms.values():
    if not arm.config_path.exists():
        print({"run":arm.run_name,"status":"not launched"});continue
    rows=arm.get_records()
    print({"run":arm.run_name,"records":len(rows),"expected":58,
           "failures":[{k:r[k] for k in ("candidate_id","blame","reason")} for r in rows if r["failed"]],
           "next":"same-input replay, paired metrics and failure/diversity sensitivity; no prompt tuning"})
